# PINN notebook (runnable)

This notebook demonstrates a minimal Physics-Informed Neural Network pipeline with:
- CSV data I/O using six inputs x = [||S_pre||, sin(theta_pre), cos(theta_pre), S11_pre, S22_pre, S12_pre].
- Stress transforms assembling x and rotating stresses for residual calculations.
- MLP backbone with data and PDE heads.
- Adaptive loss balancing via learned log-variances.
- End-to-end training, testing, and inference cells ready to run in Jupyter.


In [ ]:
import math
from pathlib import Path
from typing import Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## Data I/O
- Load from CSV with columns: S_norm, sin_theta_pre, cos_theta_pre, S11_pre, S22_pre, S12_pre (optional L_hat for labels).
- Or create synthetic data matching that schema.
- Each sample returns angles, S_pre components, a computed norm, and a target L_hat.


In [ ]:
class StressDataset(Dataset):
    def __init__(self, csv_path: Optional[str] = None, n_samples: int = 512, noise: float = 0.01):
        super().__init__()
        required_cols = ['S_norm', 'sin_theta_pre', 'cos_theta_pre', 'S11_pre', 'S22_pre', 'S12_pre']
        if csv_path and Path(csv_path).exists():
            df = pd.read_csv(csv_path)
            missing = set(required_cols) - set(df.columns)
            if missing:
                raise ValueError(f'Missing columns: {missing}')
            data = df[required_cols].to_numpy(dtype=np.float32)
            S_norm, sin_theta, cos_theta, S11, S22, S12 = data.T
            theta = np.arctan2(sin_theta, cos_theta)
            if 'L_hat' in df.columns:
                L_hat = df['L_hat'].to_numpy(dtype=np.float32)
            else:
                L_hat = 0.5 * S11 + 0.3 * S12 + 0.2 * S22
        else:
            theta_t = torch.linspace(-math.pi, math.pi, n_samples)
            sin_theta_t, cos_theta_t = torch.sin(theta_t), torch.cos(theta_t)
            S11_t = torch.sin(theta_t) * 0.6 + noise * torch.randn_like(theta_t)
            S22_t = torch.cos(theta_t) * 0.3 + noise * torch.randn_like(theta_t)
            S12_t = 0.2 * torch.sin(2 * theta_t) + noise * torch.randn_like(theta_t)
            S_norm_t = torch.sqrt(S11_t**2 + 2 * S12_t**2 + S22_t**2 + 1e-8)
            L_hat_t = 0.5 * S11_t + 0.3 * S12_t + 0.2 * S22_t
            theta, sin_theta, cos_theta = [t.numpy() for t in (theta_t, sin_theta_t, cos_theta_t)]
            S11, S22, S12, S_norm, L_hat = [t.numpy() for t in (S11_t, S22_t, S12_t, S_norm_t, L_hat_t)]
        self.theta = torch.tensor(theta, dtype=torch.float32)
        self.sin_theta = torch.tensor(sin_theta, dtype=torch.float32)
        self.cos_theta = torch.tensor(cos_theta, dtype=torch.float32)
        self.S_norm = torch.tensor(S_norm, dtype=torch.float32)
        self.S_pre = torch.stack([
            torch.tensor(S11, dtype=torch.float32),
            torch.tensor(S12, dtype=torch.float32),
            torch.tensor(S22, dtype=torch.float32),
        ], dim=1)
        self.L_hat = torch.tensor(L_hat, dtype=torch.float32)

    def __len__(self):
        return len(self.theta)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return {
            'theta': self.theta[idx],
            'sin_theta': self.sin_theta[idx],
            'cos_theta': self.cos_theta[idx],
            'S_norm': self.S_norm[idx],
            'S_pre': self.S_pre[idx],
            'L_hat': self.L_hat[idx],
        }


def save_predictions(path: str, inputs: torch.Tensor, outputs: Dict[str, torch.Tensor]) -> None:
    df = pd.DataFrame(inputs.cpu().numpy(), columns=['S_norm', 'sin_theta_pre', 'cos_theta_pre', 'S11_pre', 'S22_pre', 'S12_pre'])
    for key, value in outputs.items():
        df[key] = value.detach().cpu().numpy()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f'Saved predictions to {path}')


### CSV input/output setup
The notebook expects CSV files with columns `S_norm`, `sin_theta_pre`, `cos_theta_pre`, `S11_pre`, `S22_pre`, `S12_pre` (optionally `L_hat`).
- Training: read a CSV (or generate one below) and feed it into `StressDataset`.
- Testing: load another CSV, evaluate metrics, and export predictions to `artifacts/pinn_predictions.csv`.


In [ ]:
# Create sample CSVs for training and testing
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

theta = torch.linspace(-math.pi, math.pi, 512)
sin_theta, cos_theta = torch.sin(theta), torch.cos(theta)
S11 = torch.sin(theta) * 0.6 + 0.01 * torch.randn_like(theta)
S22 = torch.cos(theta) * 0.3 + 0.01 * torch.randn_like(theta)
S12 = 0.2 * torch.sin(2 * theta) + 0.01 * torch.randn_like(theta)
S_norm = torch.sqrt(S11**2 + 2 * S12**2 + S22**2 + 1e-8)
L_hat = 0.5 * S11 + 0.3 * S12 + 0.2 * S22

df = pd.DataFrame({
    'S_norm': S_norm.numpy(),
    'sin_theta_pre': sin_theta.numpy(),
    'cos_theta_pre': cos_theta.numpy(),
    'S11_pre': S11.numpy(),
    'S22_pre': S22.numpy(),
    'S12_pre': S12.numpy(),
    'L_hat': L_hat.numpy(),
})
train_csv = data_dir / 'train.csv'
test_csv = data_dir / 'test.csv'
df.to_csv(train_csv, index=False)
df.sample(128, random_state=0).to_csv(test_csv, index=False)
train_csv, test_csv


## Stress transforms
- Build rotation matrices.
- Pack [Sxx, Sxy, Syy] into 2x2 matrices.
- Assemble x = [|S_pre|, S_pre] used by the network.


In [ ]:
def rotation_matrix(theta: torch.Tensor) -> torch.Tensor:
    c, s = torch.cos(theta), torch.sin(theta)
    T = torch.stack([
        torch.stack([c, -s], dim=-1),
        torch.stack([s, c], dim=-1),
    ], dim=-2)
    return T


def rotate_stress(S: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    T = rotation_matrix(theta)
    return T @ S @ T.transpose(-2, -1)


def pack_stress(S_components: torch.Tensor) -> torch.Tensor:
    Sxx, Sxy, Syy = S_components.unbind(-1)
    return torch.stack([
        torch.stack([Sxx, Sxy], dim=-1),
        torch.stack([Sxy, Syy], dim=-1),
    ], dim=-2)


def stress_norm(S_components: torch.Tensor) -> torch.Tensor:
    Sxx, Sxy, Syy = S_components.unbind(-1)
    return torch.sqrt(Sxx**2 + 2 * Sxy**2 + Syy**2 + 1e-8)


def assemble_input(sin_theta: torch.Tensor, cos_theta: torch.Tensor, S_pre: torch.Tensor, S_norm: torch.Tensor):
    theta = torch.atan2(sin_theta, cos_theta)
    S_matrix = pack_stress(S_pre)
    S_rot = rotate_stress(S_matrix, theta)
    x = torch.stack([S_norm, sin_theta, cos_theta, S_pre[..., 0], S_pre[..., 2], S_pre[..., 1]], dim=-1)
    return x, S_rot


## Model
Shared MLP backbone with two heads:
- L_pred: supervised term.
- residual: PDE term driven toward zero.


In [ ]:
class StressPINN(nn.Module):
    def __init__(self, input_dim: int = 6, hidden_dim: int = 128, dropout: float = 0.05):
        super().__init__()
        layers = []
        dims = [input_dim, hidden_dim, hidden_dim, hidden_dim]
        for din, dout in zip(dims[:-1], dims[1:]):
            layers.extend([nn.Linear(din, dout), nn.GELU(), nn.Dropout(dropout)])
        self.backbone = nn.Sequential(*layers)
        self.head_L = nn.Linear(hidden_dim, 1)
        self.head_pde = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        h = self.backbone(x)
        L_pred = self.head_L(h).squeeze(-1)
        residual = self.head_pde(h).squeeze(-1)
        return {'L_pred': L_pred, 'residual': residual}


## Loss and training loop
Adaptive log-variance weights balance data and PDE terms.


In [ ]:
class AdaptiveMultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(2))

    def forward(self, losses: Tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
        balanced = []
        for i, loss in enumerate(losses):
            balanced.append(torch.exp(-self.log_vars[i]) * loss + self.log_vars[i])
        return sum(balanced)


def train_epoch(model: StressPINN, criterion: AdaptiveMultiTaskLoss, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    total = 0.0
    for batch in loader:
        sin_theta = batch['sin_theta'].to(device)
        cos_theta = batch['cos_theta'].to(device)
        S_norm = batch['S_norm'].to(device)
        S_pre = batch['S_pre'].to(device)
        target = batch['L_hat'].to(device)

        x, _ = assemble_input(sin_theta, cos_theta, S_pre, S_norm)
        preds = model(x.to(device))

        data_loss = torch.mean((preds['L_pred'] - target) ** 2)
        pde_loss = torch.mean(preds['residual'] ** 2)
        loss = criterion((data_loss, pde_loss))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(sin_theta)
    return total / len(loader.dataset)


def evaluate(model: StressPINN, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    total_data = 0.0
    total_pde = 0.0
    count = 0
    with torch.no_grad():
        for batch in loader:
            sin_theta = batch['sin_theta'].to(device)
            cos_theta = batch['cos_theta'].to(device)
            S_norm = batch['S_norm'].to(device)
            S_pre = batch['S_pre'].to(device)
            target = batch['L_hat'].to(device)

            x, _ = assemble_input(sin_theta, cos_theta, S_pre, S_norm)
            preds = model(x.to(device))

            data_loss = torch.mean((preds['L_pred'] - target) ** 2)
            pde_loss = torch.mean(preds['residual'] ** 2)
            batch_size = len(sin_theta)
            total_data += data_loss.item() * batch_size
            total_pde += pde_loss.item() * batch_size
            count += batch_size
    return {
        'data_mse': total_data / max(count, 1),
        'pde_mse': total_pde / max(count, 1),
    }


In [ ]:
# Training example
train_ds = StressDataset(csv_path=str(train_csv))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

model = StressPINN().to(device)
criterion = AdaptiveMultiTaskLoss().to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(criterion.parameters()), lr=3e-3, weight_decay=1e-4)

num_epochs = 5
for epoch in range(1, num_epochs + 1):
    loss = train_epoch(model, criterion, train_loader, optimizer)
    print(f"Epoch {epoch:02d} - loss: {loss:.4f} - log_vars: {criterion.log_vars.data.tolist()}")


## Testing module
Evaluate training precision on both the training CSV and a held-out test CSV using MSE metrics for the supervised and residual heads.


In [ ]:
# Evaluate training and test precision
test_loader = DataLoader(StressDataset(csv_path=str(test_csv)), batch_size=64, shuffle=False)
train_metrics = evaluate(model, train_loader)
test_metrics = evaluate(model, test_loader)
print('Train metrics:', train_metrics)
print('Test metrics:', test_metrics)


## Inference and export
Load a CSV for evaluation and export predictions/residuals to a CSV file.


In [ ]:
# Batch inference on test CSV
test_loader = DataLoader(StressDataset(csv_path=str(test_csv)), batch_size=64, shuffle=False)

model.eval()
batched_inputs = []
batched_preds = []
with torch.no_grad():
    for batch in test_loader:
        sin_theta = batch['sin_theta'].to(device)
        cos_theta = batch['cos_theta'].to(device)
        S_norm = batch['S_norm'].to(device)
        S_pre = batch['S_pre'].to(device)
        x, _ = assemble_input(sin_theta, cos_theta, S_pre, S_norm)
        outputs = model(x)
        batched_inputs.append(x.cpu())
        batched_preds.append({k: v.detach().cpu() for k, v in outputs.items()})

all_inputs = torch.cat(batched_inputs)
combined_preds = {k: torch.cat([p[k] for p in batched_preds]) for k in batched_preds[0]}
save_predictions('artifacts/pinn_predictions.csv', all_inputs, combined_preds)
Path('artifacts/pinn_predictions.csv').read_text().splitlines()[:5]
